# 01 · Pronóstico de pedidos por categoría

**Ejercicio principal · ~1 hora · CRISP-DM.** Elijan categoría, periodo y horizonte. Estimaremos cuántos pedidos entregados contienen al menos un artículo de esa categoría. El CSV de artículos tiene una fila por registro de artículo asociado a un pedido, pero no una columna `quantity`; por eso no permite saber cuántas unidades físicas se vendieron ni cuántas habría que vender. Tampoco registra inventario ni compras no atendidas. La demo de Colab es temporal, no un despliegue de producción.

## 1. Comprensión del negocio

Pregunta: ¿cuántos pedidos entregados incluirán productos de una categoría en los próximos periodos? Es una medida de actividad de compra por categoría, no de unidades físicas. Por ejemplo, un pedido con varios registros de artículos de la misma categoría cuenta como un pedido de esa categoría, no como varias unidades confirmadas.


In [ ]:
# Decisión del equipo: CATEGORY=None usa todas; si no, escriban una categoría de la lista.
CATEGORY = None
AGGREGATION = "M"  # "M" = mensual; "W" = semanal
HORIZON = 1  # periodos adelante: 1, 2 o 3


## 2. Comprensión de los datos y EDA

El conjunto contiene pedidos, filas de artículos, productos y una traducción de categorías. Revisaremos cobertura, estados, categorías y evolución temporal antes de transformar. Las fechas y conteos se calculan desde los CSV al ejecutar el notebook, no son resultados fijados de antemano.


In [ ]:
from urllib.error import HTTPError, URLError

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_BASE_URL = "https://raw.githubusercontent.com/JuanParias29/ML_MiniWorkshop/main/data"

def load_dataset_csv(filename):
    url = f"{DATA_BASE_URL}/{filename}"
    try:
        return pd.read_csv(url)
    except (HTTPError, URLError) as error:
        raise RuntimeError(f"No se pudo cargar {filename} desde {url}. Compruebe la conexion a internet.") from error

orders = load_dataset_csv("olist_orders_dataset.csv")
items = load_dataset_csv("olist_order_items_dataset.csv")
products = load_dataset_csv("olist_products_dataset.csv")
translation = load_dataset_csv("product_category_name_translation.csv")
print(f"Pedidos: {len(orders):,}; filas de articulos: {len(items):,}")


En `order_items`, cada fila es un registro de artículo asociado a `order_id` y `product_id`; no existe un campo que indique cuántas unidades físicas representa ni un campo de inventario. En este ejercicio contamos `order_id` distintos por categoría y periodo: cada pedido se cuenta una vez en una categoría aunque tenga varias filas de artículos de esa categoría. Así podemos pronosticar pedidos observados, pero no unidades a vender ni demanda no atendida.


In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
sales = (
    items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(translation, on="product_category_name", how="left")
    .merge(orders[["order_id", "order_status", "order_purchase_timestamp"]], on="order_id", how="inner")
)
sales["category"] = sales["product_category_name_english"].fillna(
    sales["product_category_name"]
).fillna("sin_categoria")
sales = sales.loc[sales["order_status"].eq("delivered") & sales["order_purchase_timestamp"].notna()].copy()
print(f"Rango de compras entregadas: {sales['order_purchase_timestamp'].min().date()} a {sales['order_purchase_timestamp'].max().date()}")
print(f"Categorias: {sales['category'].nunique()}; porcentaje sin categoria: {sales['product_category_name'].isna().mean():.1%}")
orders_by_category = sales.groupby("category")["order_id"].nunique().sort_values(ascending=False)
display(orders_by_category.head(10).rename("pedidos").to_frame())


### EDA: categoria y evolucion temporal

La primera figura compara pedidos que contienen cada categoría; la segunda cuenta pedidos distintos por mes. Un pedido puede aparecer en más de una categoría si incluye productos de varias categorías.


In [ ]:
top_categories = sales.groupby("category")["order_id"].nunique().nlargest(10).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top_categories.plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Diez categorías con más pedidos")
axes[0].set_xlabel("Pedidos distintos que contienen la categoría")
monthly_total = sales.drop_duplicates("order_id").set_index("order_purchase_timestamp").resample("MS")["order_id"].nunique()
monthly_total.plot(ax=axes[1], marker="o", color="darkorange")
axes[1].set_title("Pedidos distintos por mes")
axes[1].set_ylabel("Pedidos")
axes[1].set_xlabel("Mes de compra")
plt.tight_layout()
plt.show()
print(f"Mediana mensual global: {monthly_total.median():,.0f}; máximo: {monthly_total.max():,.0f} pedidos entregados")


## 3. Preparacion de los datos

Filtramos pedidos entregados con fecha válida y contamos pedidos distintos por categoría y periodo; si hay varias filas de artículos de una categoría en un pedido, cuenta una vez. Los periodos vacíos pasan a cero pedidos observados. En cada origen, rezago 1 es el periodo actual; rezagos 2–3 son los periodos anteriores. El objetivo se desplaza al horizonte elegido. Las particiones respetan el orden temporal.


In [ ]:
if AGGREGATION not in {"M", "W"}:
    raise ValueError("AGGREGATION debe ser 'M' (mensual) o 'W' (semanal).")
if HORIZON not in {1, 2, 3}:
    raise ValueError("HORIZON debe ser 1, 2 o 3 periodos.")
if CATEGORY is not None:
    if CATEGORY not in set(sales["category"]):
        raise ValueError("CATEGORY no coincide con las categorias mostradas en el EDA.")
    sales = sales.loc[sales["category"].eq(CATEGORY)].copy()
period_frequency = "M" if AGGREGATION == "M" else "W-SUN"
calendar_frequency = "MS" if AGGREGATION == "M" else "W-MON"
sales["period"] = sales["order_purchase_timestamp"].dt.to_period(period_frequency).dt.start_time
counts = sales.groupby(["category", "period"], observed=True)["order_id"].nunique()
series_parts = []
for category, values in counts.groupby(level="category"):
    values = values.droplevel("category").sort_index()
    calendar = pd.date_range(values.index.min(), values.index.max(), freq=calendar_frequency)
    complete = values.reindex(calendar, fill_value=0)
    series_parts.append(pd.DataFrame({"category": category, "date": calendar, "observed_orders": complete.to_numpy()}))
panel = pd.concat(series_parts, ignore_index=True).sort_values(["category", "date"])
grouped = panel.groupby("category")["observed_orders"]
panel["lag_1"] = panel["observed_orders"]
panel["lag_2"] = grouped.shift(1)
panel["lag_3"] = grouped.shift(2)
panel["target"] = grouped.shift(-HORIZON)
panel["target_date"] = panel.groupby("category")["date"].shift(-HORIZON)
season_lag = (12 if AGGREGATION == "M" else 52) - HORIZON
panel["seasonal_lag"] = grouped.shift(season_lag)
panel["target_month"] = panel["target_date"].dt.month
panel["horizon"] = HORIZON
panel["month_sin"] = np.sin(2 * np.pi * panel["target_month"] / 12)
panel["month_cos"] = np.cos(2 * np.pi * panel["target_month"] / 12)
panel = panel.dropna(subset=["lag_2", "lag_3", "target", "target_date"]).copy()
print(f"Observaciones de modelado: {len(panel):,}; categorias: {panel['category'].nunique()}")
fig, ax = plt.subplots(figsize=(8, 4))
panel["target"].plot.hist(bins=30, ax=ax, color="slateblue", edgecolor="white")
ax.set_title("Distribución de pedidos por categoría-periodo")
ax.set_xlabel("Pedidos distintos observados")
plt.tight_layout()
plt.show()


### Referencia metodologica

Los pronósticos ingenuos son referencias importantes y el orden temporal debe preservarse: Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*, cap. 5 ([métodos simples](https://otexts.com/fpp3/simple-methods.html)) y cap. 9 ([validación temporal](https://otexts.com/fpp3/tscv.html)). Ridge y los ensambles permiten comparar relaciones lineales y no lineales.

## 4. Modelado: comparar y seleccionar

Comparamos dos referencias (último valor y mismo periodo del año anterior), Ridge, Random Forest e HistGradientBoosting. Elegimos el menor MAE en validación, cronológicamente posterior a train. Reservamos las fechas de test para la evaluación final.


In [ ]:
dates = np.sort(panel["target_date"].unique())
if len(dates) < 10:
    raise ValueError("Se requieren al menos 10 fechas objetivo para train/validacion/prueba.")
train_end = dates[max(1, int(len(dates) * 0.60)) - 1]
validation_end = dates[max(2, int(len(dates) * 0.80)) - 1]
train = panel.loc[panel["target_date"] <= train_end].copy()
validation = panel.loc[(panel["target_date"] > train_end) & (panel["target_date"] <= validation_end)].copy()
test = panel.loc[panel["target_date"] > validation_end].copy()
if min(len(train), len(validation), len(test)) == 0:
    raise ValueError("Una particion temporal quedo vacia; elija otra agregacion o categoria.")
feature_columns = ["category", "lag_1", "lag_2", "lag_3", "horizon", "month_sin", "month_cos"]
preprocess = ColumnTransformer([
    ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["category"]),
    ("numeric", StandardScaler(), ["lag_1", "lag_2", "lag_3", "horizon", "month_sin", "month_cos"]),
])
models = {
    "Ridge": make_pipeline(preprocess, Ridge(alpha=10.0)),
    "Random Forest": make_pipeline(preprocess, RandomForestRegressor(n_estimators=160, min_samples_leaf=2, random_state=42, n_jobs=-1)),
    "HistGradientBoosting": make_pipeline(preprocess, HistGradientBoostingRegressor(max_iter=120, l2_regularization=1.0, random_state=42)),
}
for candidate in models.values():
    candidate.fit(train[feature_columns], train["target"])

def forecast_candidate(name, frame):
    if name == "Ultimo valor":
        return frame["lag_1"].to_numpy(dtype=float)
    if name == "Referencia estacional":
        return frame["seasonal_lag"].fillna(frame["lag_1"]).to_numpy(dtype=float)
    return models[name].predict(frame[feature_columns])

candidate_names = ["Ultimo valor", "Referencia estacional", *models]
validation_results = pd.DataFrame([
    {"modelo": name, "MAE validacion": mean_absolute_error(validation["target"], np.maximum(0, forecast_candidate(name, validation)))}
    for name in candidate_names
]).sort_values("MAE validacion")
display(validation_results.round(2))
best_name = validation_results.iloc[0]["modelo"]
print(f"Ganador validacion: {best_name}; train hasta {pd.Timestamp(train_end).date()}; validacion hasta {pd.Timestamp(validation_end).date()}; test comienza {test['target_date'].min().date()}")


La tabla compara MAE en validación. Reajustamos el ganador con train+validación y lo medimos una sola vez en fechas posteriores.


In [ ]:
train_validation = pd.concat([train, validation], ignore_index=True)
if best_name in models:
    best_model = models[best_name]
    best_model.fit(train_validation[feature_columns], train_validation["target"])
test = test.copy()
test["prediction"] = np.maximum(0, forecast_candidate(best_name, test))
test["naive_prediction"] = test["lag_1"]
test_scores = pd.DataFrame([
    {"modelo": "Referencia ultimo valor", "MAE prueba": mean_absolute_error(test["target"], test["naive_prediction"])},
    {"modelo": best_name, "MAE prueba": mean_absolute_error(test["target"], test["prediction"])},
])
display(test_scores.round(2))
winner_mae = float(test_scores.loc[test_scores["modelo"].eq(best_name), "MAE prueba"].iloc[0])
naive_mae = float(test_scores.loc[test_scores["modelo"].eq("Referencia ultimo valor"), "MAE prueba"].iloc[0])
if winner_mae < naive_mae:
    improvement = 100 * (naive_mae - winner_mae) / naive_mae if naive_mae else 0.0
    print(f"Resultado: en las fechas de prueba, {best_name} tuvo un error absoluto medio de {winner_mae:.1f} pedidos por categoría-periodo, {improvement:.1f}% menor que la referencia del último periodo. Sugiere mejor ajuste a este tramo histórico; no convierte el pronóstico en unidades físicas.")
elif winner_mae > naive_mae:
    print(f"Resultado: en las fechas de prueba, {best_name} tuvo un error absoluto medio de {winner_mae:.1f} pedidos por categoría-periodo, mayor que la referencia del último periodo ({naive_mae:.1f}). En este tramo, la referencia fue más precisa.")
else:
    print(f"Resultado: en las fechas de prueba, {best_name} empató con la referencia del último periodo: MAE de {winner_mae:.1f} pedidos por categoría-periodo en ambos. No se observó una reducción del error con este modelo.")


### 5. Evaluacion: observado frente a pronosticado

El MAE indica, en promedio, a cuántos pedidos por categoría-periodo se equivoca el pronóstico; menor es mejor. El gráfico compara pedidos observados con pronosticados en fechas de prueba.


In [ ]:
plot_category = CATEGORY or test["category"].value_counts().index[0]
plot_data = test.loc[test["category"].eq(plot_category)].sort_values("target_date")
if plot_data.empty:
    raise ValueError("No hay fechas de prueba para la categoria elegida.")
plt.figure(figsize=(10, 4))
plt.plot(plot_data["target_date"], plot_data["target"], marker="o", label="Observado")
plt.plot(plot_data["target_date"], plot_data["prediction"], marker="o", label=f"Pronosticado ({best_name})")
plt.title(f"Pedidos observados y pronosticados — {plot_category}")
plt.xlabel("Periodo objetivo")
plt.ylabel("Pedidos entregados distintos con esta categoría")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 6. Despliegue: demo Gradio temporal

La demo estima pedidos distintos por categoría y periodo usando el modelo elegido. La interfaz de Colab es temporal, no un despliegue de producción.


In [ ]:
import gradio as gr

categories_for_demo = sorted(panel["category"].unique().tolist())
demo_history = panel.loc[panel["category"].eq(plot_category)].sort_values("date").iloc[-1]
demo_seasonal = demo_history["seasonal_lag"] if pd.notna(demo_history["seasonal_lag"]) else demo_history["lag_1"]

def forecast_demo(category, lag_1, lag_2, lag_3, target_month, seasonal_lag_value):
    row = pd.DataFrame({
        "category": [category], "lag_1": [lag_1], "lag_2": [lag_2], "lag_3": [lag_3],
        "horizon": [HORIZON],
        "month_sin": [np.sin(2 * np.pi * target_month / 12)],
        "month_cos": [np.cos(2 * np.pi * target_month / 12)],
        "seasonal_lag": [seasonal_lag_value],
    })
    estimate = max(0.0, float(forecast_candidate(best_name, row)[0]))
    return f"Modelo elegido: {best_name}. Estimación: {estimate:.1f} pedidos entregados distintos con esta categoría para el periodo elegido; no es una cantidad de unidades físicas."

demo = gr.Interface(
    fn=forecast_demo,
    inputs=[
        gr.Dropdown(categories_for_demo, value=plot_category, label="Categoria"),
        gr.Number(value=float(demo_history["lag_1"]), minimum=0, label="Pedidos durante el periodo actual"),
        gr.Number(value=float(demo_history["lag_2"]), minimum=0, label="Pedidos hace 1 periodo"),
        gr.Number(value=float(demo_history["lag_3"]), minimum=0, label="Pedidos hace 2 periodos"),
        gr.Slider(1, 12, value=int(demo_history["target_month"]), step=1, label="Mes objetivo"),
        gr.Number(value=float(demo_seasonal), minimum=0, label="Pedidos en el mismo periodo del año anterior"),
    ],
    outputs=gr.Textbox(label="Pedidos pronosticados"),
    title="Pronóstico de pedidos por categoría",
    description=f"Ganador elegido por validación: {best_name}. El resultado cuenta pedidos, no unidades de producto.",
)
demo.launch(share=False, inline=True, prevent_thread_lock=True)
